In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as cl
import matplotlib.dates as mdates
import matplotlib
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import seaborn as sns
import netCDF4 as nc
import xarray as xr
#import cartopy.crs as ccrs
#from cartopy import feature as cfa
#import plotly.io as pio
#pio.renderers.default = 'browser'
#pio.renderers.default = 'notebook'

In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    decimal_part = decimal_year - year
    start_of_year = pd.to_datetime([f'{y}-01-01' for y in year])
    seconds_in_year = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - start_of_year).total_seconds()
    seconds = decimal_part * seconds_in_year
    return start_of_year + pd.to_timedelta(seconds, unit='s')

def datetime_to_decimal_year(datetime_):
    year = datetime_.year
    days = (datetime_.date() - datetime.date(year, 1,1)).days
    res = year + days/365.25
    return res


def FFT_res(x, si):
    n = len(x)
    x_FFT_shifted = np.fft.fftshift(np.fft.fft(x))
    x_FFT_freq_shifted = np.fft.fftshift(np.fft.fftfreq(n, d=si))
    return x_FFT_freq_shifted, x_FFT_shifted


def fft_noramlize(fft_res):
    n = len(fft_res)
    norm_fft = np.abs(fft_res)/(np.max(np.abs(fft_res)))
    return norm_fft

def line_fit(X,Y):
    P = np.polyfit(X,Y,1)
    dx = np.max(X) - np.min(X)
    X_new = np.array([np.min(X)-dx*0.05, np.max(X)+dx*0.05])
    Y_new = np.polyval(P, X_new)
    return X_new, Y_new, P

def simple_moving_average(data, window_size):
    """
    Calculates the Simple Moving Average (SMA) of a NumPy array.
    """
    # Create an array of weights, each equal to 1/window_size
    weights = np.ones(window_size) / window_size
    
    # Use np.convolve to apply the moving average filter
    # mode='valid' returns the result where the window fully overlaps the data
    sma = np.convolve(data, weights, mode='valid')
    return sma


In [ ]:
window_size = 3
window_size2 = 36


df = pd.read_excel('../../../2025-12-20_AgeModel-NewXrayPatch/age_model/interp_20251220.xlsx')
paths = list(df['path_name'].value_counts().index.values)
paths = np.flip(np.sort(paths))
for path in paths:
    df.loc[df['path_name']==path, "path_ma"] = df.loc[df['path_name']==path, 'd13c'].rolling(window_size, center=True).mean()
#df[['d13c_ma','d18o_ma']] = df[['d13c','d18o']].rolling(window_size, center=True).mean()
df_sorted = df.sort_values("age_absolute")
df_sorted['d18o_pdb'] = 0.97001*df_sorted['d18o']-29.99

df_sorted['age_ws'] = df_sorted['age_absolute'].rolling(window_size, center=True).mean()
df_sorted['age_ws2'] = df_sorted['age_absolute'].rolling(window_size2, center=True).mean()

df_sorted[['d13c_ma','d18o_ma','d18o_pdp_ma']] = df_sorted[['d13c','d18o','d18o_pdb']].rolling(window_size, center=True).mean()
df_sorted[['d13c_ma2','d18o_ma2','d18o_pdb_ma2']] = df_sorted[['d13c','d18o','d18o_pdb']].rolling(window_size2, center=True).mean()
df_sorted[['d13c_std','d18o_std', 'd18o_pdb_std']] = df_sorted[['d13c','d18o','d18o_pdb']].rolling(window_size2, center=True).std()
df_sorted['d13c_no_tectonic'] = df_sorted['d13c'] - df_sorted['d13c_ma2'].fillna(0)
df_sorted['d18o_no_tectonic'] = df_sorted['d18o'] - df_sorted['d18o_ma2'].fillna(0)
df_sorted['d18o_pdb_no_tectonic'] = df_sorted['d18o_pdb'] - df_sorted['d18o_pdb_ma2'].fillna(0)
df_sorted[['d13c_notec_ma','d18o_notec_ma','d18o_pdb_notec_ma']] = df_sorted[['d13c_no_tectonic','d18o_no_tectonic','d18o_pdb_no_tectonic']].rolling(window_size, center=True).mean()

#-----------------------RESAMPLE MEAN & STD-----------------------------------------------------

df_sorted['age_dt'] = decimal_year_to_datetime(df_sorted['age_absolute'])
df_resampled_ = df_sorted.set_index('age_dt')
df_resampled_.drop(['Name', 'Mass', 'Amp 44', 'Area 44', 'index','sample','age1','path_name',
       'path_num', 'path_ma', 'age_ws', 'age_ws2',
       'd13c_ma', 'd18o_ma', 'd13c_ma2', 'd18o_ma2', 'd13c_std', 'd18o_std'], axis=1, inplace=True)
df_resampled = df_resampled_.resample('30D').mean()

df_resampled.interpolate(inplace=True)
df_resampled[['d13c_ma','d18o_ma']] = df_resampled[['d13c','d18o']].rolling(window_size, center=True).mean()
df_resampled[['d13c_ma2','d18o_ma2']] = df_resampled[['d13c','d18o']].rolling(window_size2, center=True).mean()
df_resampled[['d13c_std','d18o_std']] = df_resampled[['d13c','d18o']].rolling(window_size2, center=True).std()

#-----------------------------------------------------------------------------------------------

df_old = df
df_resampled_old = df_resampled
df_sorted_old = df_sorted
df_sorted_old

In [ ]:
%run ../20260323_read_data.py
df_sorted

In [ ]:
df

In [ ]:
paths = df['path_name'].unique()

growth_rates = pd.DataFrame([])

for p in paths:
    df_iter = df[df['path_name'] == p]
    iter_diff = df_iter['age_final'].diff()
    growth_rates = pd.concat([growth_rates, iter_diff])
growth_rates.rename(columns={'age_final':'age_diff'}, inplace=True)
growth_rates['rates'] = 1/growth_rates['age_diff']/15


growth_rates.describe()

In [ ]:
blue = "#4287f5"
orange = "#f58442"
green = "#42f5ad"
red = "#f54254"
yellow = "#e4f000"
purple = "#f000ec"
green2 = "#00f070"
blue2 = "#00c0f0"
pink = "#e37b7b"
colors = [blue, orange, green, red, yellow, purple, green2, blue2, pink]



fig = go.Figure()
for i, path in enumerate(paths):
    df_iter = df[df['path_name']==path]
    fig.add_trace( go.Scatter(x=df_iter['age_final'], y=df_iter['d18o'], mode='markers', marker=dict(color=colors[i]), name=path, text=df_iter['Name']))
    path_ma = df_iter['d18o'].rolling(window_size, center=True).mean()
    fig.add_trace(go.Scatter(x=df_iter['age_final'].rolling(window_size, center=True).mean(), y=path_ma, mode='lines',marker=dict(color=colors[i]), showlegend=False))

fig.add_shape(type='line',
    x0=1936+(31+29+22)/365, x1=1936+(31+29+31+19)/365,
    y0=0, y1=1,  # Use yref='paper' for full height
    xref='x', yref='paper',
    line=dict(color="red", width=2, dash='dash'),
    name="EQ line"
)
fig.update_layout(width=1200, height=600, title_text="d18o", template='simple_white')
fig.show()
#fig.write_html('d18O_paths.html')

In [ ]:
dril_length = 0.15  # cm

growth_rates_df = pd.DataFrame([])

for p in paths:
    df_iter = df[df['path_name'] == p].sort_values('age_final').reset_index(drop=True)  # fix 1
    iter_diff = df_iter['age_final'].diff()
    tmp = pd.DataFrame({'age_diff': iter_diff})
    growth_rates_df = pd.concat([growth_rates_df, tmp])

growth_rates_df = growth_rates_df.dropna()                           # fix 3a
growth_rates_df = growth_rates_df[growth_rates_df['age_diff'] > 0]  # fix 3b
growth_rates_df['rates'] = dril_length / growth_rates_df['age_diff'] # fix 2

# then filter outliers as before
growth_rates = growth_rates_df[growth_rates_df['rates'] < 3]['rates'].values

In [ ]:
dril_length = 0.15 #cm

#sampling_rates_old = df_old[~df_old['age1'].isna()].groupby('path_name')['path_num'].diff().dropna()
#growth_rates_old = (sampling_rates_old*1)/10


growth_rates_df_old = pd.DataFrame([])

for p in paths:
    df_iter = df_old[df_old['path_name'] == p]
    iter_diff = df_iter['age_absolute'].diff()
    growth_rates_df_old = pd.concat([growth_rates_df_old, iter_diff])
growth_rates_df_old = growth_rates_df_old.rename(columns={'age_absolute':'age_diff'}).dropna()
#growth_rates_df_old['rates'] = 1/(growth_rates_df_old['age_diff']/dril_length)
growth_rates_df_old['rates'] = dril_length/growth_rates_df_old['age_diff']
growth_rates_old = growth_rates_df_old[growth_rates_df_old['rates']<3]['rates'].values

#sampling_rates = df[~df['age1'].isna()].groupby('path_name')['path_num'].diff().dropna()
#sampling_rates = 1/df['age_final'].diff().dropna()
#growth_rates = (sampling_rates)/(15*1)

paths = df['path_name'].unique()

growth_rates_df = pd.DataFrame([])

for p in paths:
    df_iter = df[df['path_name'] == p].sort_values('age_final').reset_index(drop=True)  # fix 1
    iter_diff = df_iter['age_final'].diff()
    tmp = pd.DataFrame({'age_diff': iter_diff})
    growth_rates_df = pd.concat([growth_rates_df, tmp])

growth_rates_df = growth_rates_df.dropna()                           # fix 3a
growth_rates_df = growth_rates_df[growth_rates_df['age_diff'] > 0]  # fix 3b
growth_rates_df['rates'] = dril_length / growth_rates_df['age_diff'] # fix 2

# then filter outliers as before
growth_rates = growth_rates_df[growth_rates_df['rates'] < 3]['rates'].values


# Drill Samples were 1.5 mm ?????

#####################################

fig, axes = plt.subplots(1,2, figsize=(12,4), width_ratios=(2,1), sharey=True, gridspec_kw={'wspace':0})

ax = axes[0]
ax.plot(growth_rates_old, '.',markersize=3, color='#D9710B', linewidth=0.75, label='old age model')
ax.plot(growth_rates, '.-', markersize=3, color='#0E35C4' , linewidth=0.3, label='new age model')
window_size = 10
growth_rates_filtered = simple_moving_average(growth_rates, window_size)
ax.plot(growth_rates_filtered, '-', markersize=3, color='black' , linewidth=1.5, label='new age model')
ax.grid()
ax.legend()

ax = axes[1]

mu = np.mean(growth_rates_old)
sigma = np.std(growth_rates_old)
bin_count = 30

count, bins, ignored = ax.hist(growth_rates_old, bins=np.arange(1,bin_count)/10, rwidth=0.7,color='#824407', alpha=0.5, align='left',orientation='horizontal' ,density=True, label='old')
ax.plot(1/(sigma*np.sqrt(2*np.pi))*np.exp(-(bins-mu)**2 / (2*sigma**2)),bins,linewidth=2, color='#D9710B')


mu = np.mean(growth_rates)
sigma = np.std(growth_rates)
bin_count = 30

count, bins, ignored = ax.hist(growth_rates, bins=np.arange(1,bin_count)/10, rwidth=0.7, color='#130561', alpha=0.5, align='mid',orientation='horizontal' ,density=True, label='new')
ax.plot(1/(sigma*np.sqrt(2*np.pi))*np.exp(-(bins-mu)**2 / (2*sigma**2)),bins,linewidth=2, color='#0E35C4')

ax.set_yticks(np.arange(1,bin_count+1,3)/10)
ax.set_ylim(1/10,bin_count/10+1/10)
ax.grid()
ax.legend()

ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")
ax.set_ylabel("Growth Rate [cm/a]")


#fig.suptitle('Sampling Rate Distribution')

axes[1].set_title('Growth Rate Distribution')
axes[0].set_title('Growth Rate over Sample Age')
axes[0].set_ylabel('Growth Rate [cm/a]')
axes[0].set_xlabel('Sample Index ~ Time')
ax.set_xlabel('Probability')


#fig.savefig('sampling_rate_distribution_compare.pdf')

In [ ]:
dril_length = 0.15 #cm

#sampling_rates_old = df_old[~df_old['age1'].isna()].groupby('path_name')['path_num'].diff().dropna()
#growth_rates_old = (sampling_rates_old*1)/10


growth_rates_df_old = pd.DataFrame([])

for p in paths:
    df_iter = df_old[df_old['path_name'] == p]
    iter_diff = df_iter['age_absolute'].diff()
    growth_rates_df_old = pd.concat([growth_rates_df_old, iter_diff])
growth_rates_df_old = growth_rates_df_old.rename(columns={'age_absolute':'age_diff'}).dropna()
#growth_rates_df_old['rates'] = 1/(growth_rates_df_old['age_diff']/dril_length)
growth_rates_df_old['rates'] = dril_length/growth_rates_df_old['age_diff']
growth_rates_old = growth_rates_df_old[growth_rates_df_old['rates']<3]['rates'].values

#sampling_rates = df[~df['age1'].isna()].groupby('path_name')['path_num'].diff().dropna()
#sampling_rates = 1/df['age_final'].diff().dropna()
#growth_rates = (sampling_rates)/(15*1)

paths = df['path_name'].unique()

growth_rates_df = pd.DataFrame([])

for p in paths:
    df_iter = df[df['path_name'] == p].sort_values('age_final').reset_index(drop=True)  # fix 1
    iter_diff = df_iter['age_final'].diff()
    tmp = pd.DataFrame({'age_diff': iter_diff})
    growth_rates_df = pd.concat([growth_rates_df, tmp])

growth_rates_df = growth_rates_df.dropna()                           # fix 3a
growth_rates_df = growth_rates_df[growth_rates_df['age_diff'] > 0]  # fix 3b
growth_rates_df['rates'] = dril_length / growth_rates_df['age_diff'] # fix 2

# then filter outliers as before
growth_rates = growth_rates_df[growth_rates_df['rates'] < 3]['rates'].values


# Drill Samples were 1.5 mm ?????

#####################################

fig, axes = plt.subplots(1,2, figsize=(10,4), width_ratios=(2,1), sharey=True, gridspec_kw={'wspace':0})

ax = axes[0]
#ax.plot(growth_rates, '.-', markersize=3, color='#0E35C4' , linewidth=0.3, label='new age model')
ax.plot(growth_rates, '.', markersize=3, color='#0E35C4' ,  label='new age model')
window_size = 10
growth_rates_filtered = simple_moving_average(growth_rates, window_size)
ax.plot(growth_rates_filtered, '-', markersize=3, color='black' , linewidth=1.5, label='new age model')
ax.grid()
#ax.legend()

ax = axes[1]


mu = np.mean(growth_rates)
sigma = np.std(growth_rates)
bin_count = 30

count, bins, ignored = ax.hist(growth_rates, bins=np.arange(1,bin_count)/10, rwidth=0.7, color='#130561', alpha=0.5, align='mid',orientation='horizontal' ,density=True)
ax.plot(1/(sigma*np.sqrt(2*np.pi))*np.exp(-(bins-mu)**2 / (2*sigma**2)),bins,linewidth=2, color='#0E35C4', label='PDF')

ax.set_yticks(np.arange(1,bin_count+1,3)/10)
ax.set_ylim(1/10,bin_count/10+1/10)
ax.grid()
ax.legend()

ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")
ax.set_ylabel("Growth Rate [cm/a]")


#fig.suptitle('Sampling Rate Distribution')

axes[1].set_title('Growth Rate Distribution')
axes[0].set_title('Growth Rate over Sample Age')
axes[0].set_ylabel('Growth Rate [cm/a]')
axes[0].set_xlabel('Sample Index ~ Time')
ax.set_xlabel('Probability')


fig.savefig('sampling_rate_distribution.pdf')

In [ ]:
#np.save('growth_rate.csv', growth_rates_filtered)

In [ ]:
fig, ax = plt.subplots(1,figsize=(8,5))

ax.plot(df_sorted_old['age_absolute'], df_sorted_old['d13c'], '.', color='#F7A559', alpha=0.5)
ax.plot(df_sorted_old['age_ws'], df_sorted_old['d13c_ma'],  color='#F27E1D', alpha=0.8, label='old model')
ax.plot(df_sorted_old['age_ws2'], df_sorted_old['d13c_ma2'], color='#E3470E')



ax.plot(df_sorted['age_final'], df_sorted['d13c'], '.', color='#27F535', alpha=0.5)
ax.plot(df_sorted['age_ws'], df_sorted['d13c_ma'], color='#05700C', alpha=0.8, label='new model')
ax.plot(df_sorted['age_ws2'], df_sorted['d13c_ma2'], color='#011803')

ax.legend()
ax.set_ylabel(r'$\delta^{13}C$')

#fig.savefig('tsa_d13c_compare.png')

In [ ]:
dark_band_thickness = dark_bands['path_num_end'] - dark_bands['path_num_start'].values
dark_band_growth_rate = (1.5*dark_band_thickness * dril_length)
dark_band_thickness_idx= dark_bands[['index_end','index_start']].mean(axis=1)
dark_band_thickness_date = dark_bands[['age_final_end','age_final_start']].mean(axis=1)

In [ ]:
for p in paths:
    df_iter = df[df['path_name'] == p].sort_values('age_final').reset_index(drop=True) 
    age_diff = df_iter[~df_iter['age1'].isna()]['age_final'].diff()
    sample_diff = df_iter[~df_iter['age1'].isna()]['path_num'].diff()
    

In [ ]:
df_iter[~df_iter['age1'].isna()].loc[1:, 'age_final' ]

In [ ]:
paths = df['path_name'].unique()

growth_bands = pd.DataFrame([])

for p in paths:
    df_iter = df[df['path_name'] == p].sort_values('age_final').reset_index(drop=True)   
    tmp = df_iter[~df_iter['age1'].isna()][['path_num', 'age_final']].diff().dropna()
    tmp.rename(columns={'path_num':'sample_diff', 'age_final':'age_diff'}, inplace=True)
    tmp['age_final'] =  df_iter[~df_iter['age1'].isna()].loc[1:, 'age_final' ]
    growth_bands = pd.concat([growth_bands, tmp])

growth_bands.reset_index(drop=True, inplace=True)
growth_bands

In [ ]:
blue = "#4287f5"
orange = "#f58442"
green = "#42f5ad"
red = "#f54254"
yellow = "#e4f000"
purple = "#f000ec"
green2 = "#00f070"
blue2 = "#00c0f0"
pink = "#e37b7b"
colors = [blue, orange, green, red, yellow, purple, green2, blue2, pink]


fig = make_subplots(specs=[[{"secondary_y": True}]])

for i, path in enumerate(paths):
    df_iter = df[df['path_name']==path]
    fig.add_trace( go.Scatter(x=df_iter['age_final'], y=df_iter['d13c'], mode='markers', marker=dict(color=colors[i]), name=path, text=df_iter['Name']))
    path_ma = df_iter['d13c'].rolling(window_size, center=True).mean()
    fig.add_trace(go.Scatter(x=df_iter['age_final'].rolling(window_size, center=True).mean(), y=path_ma, mode='lines',marker=dict(color=colors[i]), showlegend=False))

fig.add_shape(type='line',
    x0=1936+(31+29+22)/365, x1=1936+(31+29+31+19)/365,
    y0=0, y1=1,  # Use yref='paper' for full height
    xref='x', yref='paper',
    line=dict(color="red", width=2, dash='dash'),
    name="EQ line"
)

fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.add_trace(go.Scatter(x=growth_bands['age_final'], y=growth_bands['sample_diff'], name='band thickness',  mode='lines', line_color='#525252'), secondary_y=True)
fig.add_trace(go.Scatter(x=dark_band_thickness_date, y=dark_band_thickness, name='dark band thickness',  mode='lines', line_color='black'), secondary_y=True)



fig.update_layout(width=1200, height=600, title_text="d13C", template='simple_white')
fig.show()
#fig.write_html('d18O_paths.html')






In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,4), width_ratios=(2,1), sharey=True, gridspec_kw={'wspace':0})

ax = axes[0]
ax.plot(growth_rates_old, '-',markersize=3, color='#D9710B', linewidth=0.75, label='old age model')
ax.plot(growth_rates, '-', markersize=3, color='#0E35C4' , linewidth=0.75, label='new age model')

#ax2 =ax.twinx()
ax.plot(dark_band_thickness_idx, dark_band_growth_rate, '.-', color='#1CA333', linewidth=0.9, label='dark band thicknes')

ax.grid()
ax.legend()


ax = axes[1]

mu = np.mean(growth_rates_old)
sigma = np.std(growth_rates_old)
bin_count = 22

count, bins, ignored = ax.hist(growth_rates_old, bins=np.arange(1,bin_count)/10, rwidth=0.7,color='#824407', alpha=0.5, align='left',orientation='horizontal' ,density=True, label='old')
ax.plot(1/(sigma*np.sqrt(2*np.pi))*np.exp(-(bins-mu)**2 / (2*sigma**2)),bins,linewidth=2, color='#D9710B')


mu = np.mean(growth_rates)
sigma = np.std(growth_rates)
bin_count = 30

count, bins, ignored = ax.hist(growth_rates, bins=np.arange(1,bin_count)/10, rwidth=0.7, color='#130561', alpha=0.5, align='mid',orientation='horizontal' ,density=True, label='new')
ax.plot(1/(sigma*np.sqrt(2*np.pi))*np.exp(-(bins-mu)**2 / (2*sigma**2)),bins,linewidth=2, color='#0E35C4')

mu = np.mean(dark_band_growth_rate)
sigma = np.std(dark_band_growth_rate)
bin_count = 30

count, bins, ignored = ax.hist(dark_band_growth_rate, bins=np.arange(1,bin_count)/10, rwidth=0.7,color='#77E487', alpha=0.5, align='left',orientation='horizontal' ,density=True, label='dark band x2')
ax.plot(1/(sigma*np.sqrt(2*np.pi))*np.exp(-(bins-mu)**2 / (2*sigma**2)),bins,linewidth=2, color='#1CA333')


ax.set_yticks(np.arange(1,bin_count+1,2)/10)
ax.set_ylim(1/10,bin_count/10+1/10)
ax.grid()
ax.legend()

ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")
ax.set_ylabel("Growth Rate [cm/a]")


#fig.suptitle('Sampling Rate Distribution')

axes[1].set_title('Growth Rate Distribution')
axes[0].set_title('Growth Rate over Sample Age')
axes[0].set_ylabel('Growth Rate [cm/a]')
axes[0].set_xlabel('Sample Index ~ Time')
ax.set_xlabel('Probability')
